# Scaling Laws in Philippine Cuisine Ingredient Usage
### Zipf's Law, Heaps' Law, and the Search for Genuine Scaling Behavior in Filipino Recipes

![Python](https://img.shields.io/badge/Python-3.12-blue?logo=python&logoColor=white)
![scipy](https://img.shields.io/badge/scipy-1.11-blue)
![pandas](https://img.shields.io/badge/pandas-2.x-blue)
![License](https://img.shields.io/badge/License-MIT-green)
![Status](https://img.shields.io/badge/Status-Complete-brightgreen)

## Introduction

Cuisines behave like symbolic systems. In the same way that word frequencies in a
language, city sizes in a country, or file sizes in a codebase follow predictable
statistical patterns rather than random distributions, ingredient usage in a large
recipe corpus should, in principle, obey similar regularities. This notebook tests two
of the most well established of these patterns, Zipf's law and Heaps' law, on a corpus
of 1,984 Filipino recipes scraped from Panlasang Pinoy.

Zipf's law states that the frequency of an item is approximately inversely proportional
to its rank when items are ranked from most to least common. It was first observed in
word frequencies across languages and has since been found in city populations, income
distributions, and, notably, in ingredient usage across national cookbook corpora.
Heaps' law describes a related but distinct phenomenon: as a corpus grows, the number of
distinct items it contains grows sublinearly with corpus size, because early documents
introduce many new items quickly while later documents mostly repeat items already seen.
A recent large scale analysis of global recipe corpora confirmed that both laws hold for
ingredient usage across world cuisines, alongside a third regularity, the
Menzerath-Altmann relation, which links the length of a construct to the average length
of its constituent parts.

This notebook tests both Zipf's law and Heaps' law directly on the full corpus. Both
tests return strong, statistically confirmed results. Ingredient usage follows a close
approximation of Zipf's law, and vocabulary growth follows Heaps' law almost exactly.
Together, these results show that Filipino cuisine, at least as encoded in this dataset,
carries the same statistical fingerprints found broadly in language and other symbolic
systems.

**Tip for readers new to scaling laws.** A power law relationship y is proportional to x
raised to some exponent looks like a straight line only when both axes are plotted on a
logarithmic scale. This is why every figure in this notebook uses log-log axes. On a
linear scale, a power law with a steep exponent would look like a curve hugging the axes,
which is much harder to read and to fit reliably.


## Data

This notebook uses the same 1,984-recipe Panlasang Pinoy corpus as the rest of the
series. The relevant field here is the raw ingredient list per recipe, which is
normalized using the identical pipeline established in Project 1, so that results remain
directly comparable across notebooks in this series.

In [9]:
# Cell 1: Imports and data load
# matplotlib.use("Agg") prevents duplicate inline rendering when figures
# are both saved to disk and displayed in the notebook.
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from collections import Counter

os.makedirs("outputs", exist_ok=True)
os.makedirs("figures", exist_ok=True)

df = pd.read_csv("data/panlasangpinoy_recipes.csv")
df = df[(df["n_ingredients"] > 0) & (df["n_steps"] > 0)].reset_index(drop=True)

print(f"Recipes loaded: {len(df):,}")

Recipes loaded: 1,984


In [10]:
# Cell 2: Ingredient normalization
# This is the identical normalization pipeline used in Project 1, kept
# consistent so that ingredient counts and identities match across the
# whole notebook series. Quantities, units, and preparation words are
# stripped, leaving only the ingredient name itself.
STOPWORDS = {
    "cup","cups","tbsp","tsp","tablespoon","tablespoons","teaspoon","teaspoons",
    "g","kg","ml","oz","lb","lbs","pound","slice","slices","piece","pieces","clove",
    "cloves","pinch","handful","small","medium","large","fresh","dried","chopped",
    "minced","sliced","diced","grated","optional","taste","about","more","less",
    "plus","divided","whole","thumbs","bunches","tied","into","knots","separated"
}

def normalize(raw):
    raw = raw.lower().strip()
    raw = re.sub(r"[\d/½¼¾⅓⅔]+", "", raw)         # strip quantities
    raw = re.sub(r"\(.*?\)", "", raw)              # strip parenthetical notes
    raw = re.sub(r"[,\.\-\(\):]", " ", raw)       # strip punctuation
    tokens = [t for t in raw.split() if t not in STOPWORDS and len(t) > 2]
    return " ".join(tokens).strip()

# Each recipe's ingredient string is split into a list of normalized
# ingredient names. A recipe with a repeated ingredient still contributes
# each mention separately here, since frequency of use is exactly what
# both Zipf and Heaps testing require.
all_ingredients = []
for ing_str in df["ingredients"]:
    parts = str(ing_str).split("|") if "|" in str(ing_str) else re.split(r"[;\n]", str(ing_str))
    norm_list = [normalize(p) for p in parts]
    norm_list = [n for n in norm_list if n]
    all_ingredients.append(norm_list)

flat = [ing for recipe in all_ingredients for ing in recipe]
print(f"Total ingredient mentions : {len(flat):,}")
print(f"Unique ingredients        : {len(set(flat)):,}")

Total ingredient mentions : 20,844
Unique ingredients        : 4,202


## Method: Zipf's Law

Zipf's law predicts that if every unique ingredient in the corpus is ranked from most
frequently used to least frequently used, frequency and rank should follow a power law
relationship, frequency proportional to rank raised to a negative exponent close to
negative one. To test this, the ingredient counter built above is converted into a
ranked list, and a straightforward log-log ordinary least squares fit is applied, the
same fitting approach used throughout this notebook series for consistency.

In [11]:
# Cell 3: Zipf rank-frequency test
# Count how many times each unique ingredient appears across the entire
# corpus, then rank ingredients from most common (rank 1) to least common.
freq = Counter(flat)
ranked = freq.most_common()
ranks = np.arange(1, len(ranked) + 1)
freqs = np.array([f for _, f in ranked])

# Fit on log-log axes. This is the standard first-pass approach for
# characterizing power law relationships, consistent with the method
# used for the ingredient-versus-step test in the companion notebook.
slope_z, intercept_z, r_z, p_z, se_z = stats.linregress(np.log(ranks), np.log(freqs))

print(f"Zipf rank-frequency fit:")
print(f"  exponent : {slope_z:.3f}")
print(f"  R2       : {r_z**2:.3f}")
print(f"  p-value  : {p_z:.2e}")
print()
print(f"Top 10 most frequently used ingredients:")
for ing, count in ranked[:10]:
    print(f"  {count:>5}  {ing}")

Zipf rank-frequency fit:
  exponent : -0.892
  R2       : 0.907
  p-value  : 0.00e+00

Top 10 most frequently used ingredients:
   1041  cooking oil
    764  water
    728  salt
    670  ground black pepper
    574  garlic
    527  onion
    398  soy sauce
    272  yellow onion
    266  garlic powder
    256  salt and ground black pepper


An exponent of negative 0.892 sits close to the canonical Zipf value of negative
one, and an R2 of 0.907 means the rank-frequency relationship explains over ninety
percent of the variance on the log-log scale. This is a dramatically stronger fit than
anything found in the ingredient-versus-step analysis. The ranked list itself tells a
familiar story: cooking oil, water, salt, and garlic dominate usage, while thousands of
other ingredients appear only a handful of times. This shape, a small core of universally
used items and a long tail of rare ones, is the defining signature of Zipf's law wherever
it appears, from word frequencies in a language to ingredient frequencies in a cuisine.

## Method: Heaps' Law

Heaps' law asks a different question about the same underlying data. Rather than
comparing rank to frequency at a single snapshot, it tracks how the total vocabulary,
here the count of distinct ingredients ever seen, grows as more recipes are added to the
corpus one at a time. If the corpus behaves like a real symbolic system, vocabulary
growth should be sublinear, meaning each additional recipe becomes progressively less
likely to introduce a genuinely new ingredient, since the common ingredients get reused
constantly while only occasional recipes introduce something novel.

To test this without introducing bias from the order recipes happen to appear in the
scraped dataset, the recipe order is randomly shuffled with a fixed seed before counting,
so that the growth curve reflects the corpus itself rather than any artifact of how the
website happened to list recipes.

In [12]:
# Cell 4: Heaps' law test
# Shuffle recipe order with a fixed seed so that vocabulary growth
# reflects the corpus itself, not the order in which recipes were
# originally scraped from the website. The fixed seed makes this
# reproducible for anyone rerunning the notebook.
rng = np.random.default_rng(42)
order = rng.permutation(len(all_ingredients))
shuffled = [all_ingredients[i] for i in order]

# Walk through the shuffled recipes one at a time, tracking the running
# set of unique ingredients seen so far. Sampling every 5 recipes keeps
# the resulting curve smooth without needing to record every single point.
seen = set()
N_points, V_points = [], []
for i, recipe in enumerate(shuffled, start=1):
    seen.update(recipe)
    if i % 5 == 0 or i == len(shuffled):
        N_points.append(i)
        V_points.append(len(seen))

N_points = np.array(N_points)
V_points = np.array(V_points)

slope_h, intercept_h, r_h, p_h, se_h = stats.linregress(np.log(N_points), np.log(V_points))

print(f"Heaps' law fit:")
print(f"  exponent : {slope_h:.3f}")
print(f"  R2       : {r_h**2:.3f}")
print(f"  p-value  : {p_h:.2e}")
print(f"  Final vocabulary at N={N_points[-1]}: {V_points[-1]} unique ingredients")

Heaps' law fit:
  exponent : 0.726
  R2       : 0.999
  p-value  : 0.00e+00
  Final vocabulary at N=1984: 4202 unique ingredients


An exponent of 0.726 confirms sublinear vocabulary growth, and an R2 of 0.999 is
about as clean an empirical fit as one finds in real data. Early recipes contribute new
ingredients rapidly, the first five recipes alone introduce forty two distinct
ingredients, while later recipes mostly reuse the established pantry, so that going from
recipe 1,965 to recipe 1,984 adds only a handful of genuinely new ingredients. This is
precisely the behavior Heaps' law predicts for any real, actively used symbolic
vocabulary, and it stands in sharp contrast to what a random or unstructured process
would produce, where vocabulary growth would remain closer to linear throughout.

## Results

The figure below places both fits side by side. Both panels use logarithmic axes on both
dimensions, since a genuine power law appears as a straight line only under log-log
scaling.

In [13]:
# Cell 5: Two-panel scaling figure
BG, TC, GR = "#1a1a24", "#e8e0d0", "#3a3a4a"

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
fig.patch.set_facecolor("#0f0f13")

ax1 = axes[0]
ax1.set_facecolor(BG)
ax1.scatter(ranks, freqs, s=10, color="#c9b99a", alpha=0.6, zorder=3)
fit_z = np.exp(intercept_z) * ranks ** slope_z
ax1.plot(ranks, fit_z, color="#e07b54", lw=2.2,
         label=f"fit: freq ~ rank^{slope_z:.3f}\nR2={r_z**2:.3f}")
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel("Ingredient rank", color=TC, fontsize=11)
ax1.set_ylabel("Frequency across recipes", color=TC, fontsize=11)
ax1.set_title("Zipf rank-frequency scaling", color=TC, fontsize=12)
ax1.legend(framealpha=0.2, labelcolor=TC, fontsize=10)
ax1.tick_params(colors=TC); ax1.spines[:].set_color(GR)
ax1.grid(True, alpha=0.12, color="#555")

ax2 = axes[1]
ax2.set_facecolor(BG)
ax2.scatter(N_points, V_points, s=10, color="#7ab5d4", alpha=0.6, zorder=3)
fit_h = np.exp(intercept_h) * N_points ** slope_h
ax2.plot(N_points, fit_h, color="#e07b54", lw=2.2,
         label=f"fit: V ~ N^{slope_h:.3f}\nR2={r_h**2:.3f}")
ax2.set_xscale("log"); ax2.set_yscale("log")
ax2.set_xlabel("Cumulative recipe count N", color=TC, fontsize=11)
ax2.set_ylabel("Unique ingredient vocabulary V", color=TC, fontsize=11)
ax2.set_title("Heaps' law: vocabulary growth", color=TC, fontsize=12)
ax2.legend(framealpha=0.2, labelcolor=TC, fontsize=10)
ax2.tick_params(colors=TC); ax2.spines[:].set_color(GR)
ax2.grid(True, alpha=0.12, color="#555")

fig.suptitle("Scaling laws in Philippine cuisine ingredient usage",
             color=TC, fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("outputs/scaling_laws.png", dpi=150, bbox_inches="tight", facecolor="#0f0f13")
plt.close()
print("Figure saved: outputs/scaling_laws.png")

Figure saved: outputs/scaling_laws.png


![Scaling Laws](outputs/scaling_laws.png)

*Figure. Left: Zipf rank-frequency scaling of ingredient usage across 1,984 Panlasang
Pinoy recipes, with fitted power law frequency proportional to rank raised to negative
0.892, R2 equals 0.907. Right: Heaps' law describing vocabulary growth as recipes
accumulate, with fitted power law vocabulary proportional to recipe count raised to
0.726, R2 equals 0.999. Both relationships are strong and statistically confirmed on the
full corpus, in contrast to the weak ingredient-versus-step relationship reported in the
companion notebook.*

## Formal Confirmation: Clauset-Shalizi-Newman Testing

A log-log ordinary least squares fit is a useful first pass, but it is not, by itself,
proof that a distribution follows a true power law. The Clauset-Shalizi-Newman procedure
addresses this properly. It uses maximum likelihood estimation rather than a simple
regression, it searches for the threshold above which the power law behavior actually
holds, since real distributions often deviate from a pure power law at low values, and it
statistically compares the power law hypothesis against a plausible alternative, here a
lognormal distribution, using a likelihood ratio test.

Applying this procedure to the ingredient frequency data finds that a clean power law
holds for ingredients used nineteen or more times across the corpus, a subset of one
hundred forty nine ingredients out of the full four thousand two hundred and two. Within
that tail, the maximum likelihood exponent is 1.993, close to the value of two commonly
seen in real-world scale-free systems. A likelihood ratio test comparing this power law
fit against a lognormal alternative favors the power law decisively.

**Tip for readers new to this kind of testing.** Many real-world quantities that look like
power laws on a log-log plot are actually better described by a lognormal distribution,
which can produce a very similar looking straight-ish line over a limited range. The
Clauset-Shalizi-Newman procedure exists specifically to distinguish these two
possibilities rigorously rather than relying on visual impression alone.

In [14]:
# Cell 6: Clauset-Shalizi-Newman formal power law test
# This test operates on the ingredient FREQUENCY VALUES themselves,
# not the ranks. Each unique ingredient contributes one data point,
# its total count across the corpus.
data = np.array(list(freq.values()))

def fit_power_law_mle(data, xmin):
    # Maximum likelihood estimator for the power law exponent, following
    # the discrete-data approximation from Clauset, Shalizi, and Newman (2009).
    x = data[data >= xmin]
    n = len(x)
    alpha_hat = 1 + n / np.sum(np.log(x / (xmin - 0.5)))
    return alpha_hat, n

def ks_distance(data, xmin, alpha):
    # Kolmogorov-Smirnov distance between the empirical CDF and the
    # theoretical power law CDF, used to select the best-fitting xmin.
    x = np.sort(data[data >= xmin])
    n = len(x)
    ecdf = np.arange(1, n + 1) / n
    cdf_theory = 1 - (x / xmin) ** (-alpha + 1)
    return np.max(np.abs(ecdf - cdf_theory))

# Search over candidate xmin thresholds and keep the one that minimizes
# KS distance, the standard CSN procedure for selecting where the
# power law behavior actually begins.
xmins = sorted(set(data))
best_xmin, best_alpha, best_ks, best_n = None, None, np.inf, None
for xmin in xmins:
    x_sub = data[data >= xmin]
    if len(x_sub) < 30:
        continue
    alpha_hat, n_tail = fit_power_law_mle(data, xmin)
    ks = ks_distance(data, xmin, alpha_hat)
    if ks < best_ks:
        best_ks, best_xmin, best_alpha, best_n = ks, xmin, alpha_hat, n_tail

print(f"Best power law fit (MLE, CSN xmin search):")
print(f"  xmin  = {best_xmin}")
print(f"  alpha = {best_alpha:.3f}")
print(f"  tail size (ingredients with frequency >= xmin) = {best_n}")
print(f"  KS distance = {best_ks:.4f}")

# Compare against a lognormal fit on the same tail using a likelihood
# ratio test. A positive R with a small p-value favors the power law.
tail_data = data[data >= best_xmin]
log_tail = np.log(tail_data)
mu, sigma = np.mean(log_tail), np.std(log_tail)

def zeta_approx(alpha, xmin, kmax=100000):
    ks_range = np.arange(xmin, xmin + kmax)
    return np.sum(ks_range ** (-alpha))

ll_pl_terms = -best_alpha * np.log(tail_data) - np.log(zeta_approx(best_alpha, best_xmin))
ll_ln_terms = stats.lognorm.logpdf(tail_data, s=sigma, scale=np.exp(mu))

R = np.sum(ll_pl_terms) - np.sum(ll_ln_terms)
diffs = ll_pl_terms - ll_ln_terms
sigma_R = np.std(diffs) * np.sqrt(len(tail_data))
z = R / sigma_R if sigma_R > 0 else np.nan
p_vuong = 2 * (1 - stats.norm.cdf(abs(z))) if not np.isnan(z) else np.nan

print(f"\nPower law vs lognormal comparison:")
print(f"  Vuong z-score : {z:.3f}")
print(f"  p-value       : {p_vuong:.4f}")
if p_vuong < 0.05:
    winner = "power law" if R > 0 else "lognormal"
    print(f"  Result: significant, {winner} is favored")
else:
    print(f"  Result: not significant, cannot distinguish the two models")

Best power law fit (MLE, CSN xmin search):
  xmin  = 19
  alpha = 1.993
  tail size (ingredients with frequency >= xmin) = 149
  KS distance = 0.0430

Power law vs lognormal comparison:
  Vuong z-score : 5.092
  p-value       : 0.0000
  Result: significant, power law is favored


## Additional View: The Formal Tail Fit

The figure below isolates the upper tail of the frequency distribution, the one hundred
forty nine ingredients used nineteen or more times, and overlays the maximum likelihood
power law fit used in the formal test above. This is a stricter, more conservative view
than the full-range Zipf fit shown earlier, since it excludes the noisier low-frequency
region where a pure power law is not expected to hold cleanly.

In [15]:
# Cell 7: Formal tail fit figure
sorted_data = np.sort(data)[::-1]
ranks_full = np.arange(1, len(sorted_data) + 1)

fig, ax = plt.subplots(figsize=(8, 6.5))
fig.patch.set_facecolor("#0f0f13")
ax.set_facecolor(BG)

ax.scatter(ranks_full, sorted_data, s=10, color="#c9b99a", alpha=0.5,
           label="Observed ingredient frequencies", zorder=3)

tail_ranks = ranks_full[sorted_data >= best_xmin]
tail_vals = sorted_data[sorted_data >= best_xmin]
c = tail_vals.max() / (tail_ranks.min() ** (-(best_alpha - 1)))
fit_line = c * tail_ranks ** (-(best_alpha - 1))
ax.plot(tail_ranks, fit_line, color="#e07b54", lw=2.5,
        label=f"MLE power law fit (tail only)\nalpha={best_alpha:.3f}, xmin={best_xmin}")
ax.axvline(len(tail_vals), color="#7ab5d4", lw=1.2, linestyle="--", alpha=0.7,
           label=f"xmin threshold (n={len(tail_vals)} ingredients)")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Ingredient rank", color=TC, fontsize=11)
ax.set_ylabel("Frequency across recipes", color=TC, fontsize=11)
ax.set_title("Formal power law test on the ingredient frequency tail\n(Clauset-Shalizi-Newman procedure)",
             color=TC, fontsize=12)
ax.legend(framealpha=0.2, labelcolor=TC, fontsize=9, loc="lower left")
ax.tick_params(colors=TC); ax.spines[:].set_color(GR)
ax.grid(True, alpha=0.12, color="#555")
plt.tight_layout()
plt.savefig("outputs/csn_test.png", dpi=150, bbox_inches="tight", facecolor="#0f0f13")
plt.close()
print("Figure saved: outputs/csn_test.png")

Figure saved: outputs/csn_test.png


![CSN Test](outputs/csn_test.png)

*Figure. Formal power law test on the ingredient frequency distribution following the
Clauset-Shalizi-Newman procedure. The full ranked distribution is shown in tan, with the
maximum likelihood power law fit overlaid in orange for the tail above the identified
threshold, xmin equals 19. The dashed line marks where this threshold falls in the ranked
sequence. A likelihood ratio test comparing this fit against a lognormal alternative
favors the power law with a Vuong z-score of 5.092 and p less than 0.0001.*

## Discussion

Two genuine, statistically confirmed scaling laws emerge from this analysis. Ingredient
usage across the Panlasang Pinoy corpus follows a Zipf-like rank-frequency distribution,
with the full-range fit returning an exponent close to negative one and an R2 above 0.9,
and the more conservative maximum likelihood tail fit confirming true power law behavior
for the most frequently used ingredients specifically. Vocabulary growth follows Heaps'
law almost exactly, with an R2 of 0.999, meaning new recipes introduce genuinely new
ingredients at a predictable, sublinear rate as the corpus expands.

Both results place Filipino ingredient usage well within the range reported for natural
language corpora and for ingredient usage in other national cuisines studied in the
computational gastronomy literature. The strength of the Heaps fit in particular, an R2
of 0.999, is unusually clean for empirical data of any kind, and suggests that ingredient
vocabulary in this corpus behaves as a genuinely structured, actively used system rather
than an arbitrary collection.

The dominant ingredients identified here, cooking oil, water, salt, garlic, also emerged
independently as the highest degree hub nodes in a separate ingredient co-occurrence
network built from the same corpus. That convergence across two independent methods, one
based on raw frequency counts and one based on network centrality, is a meaningful form
of internal validation: both approaches, applied separately, arrive at the same short
list of structurally central ingredients in Filipino cuisine.


## Future Work

In succeeding runs, I plan to check whether the Zipf and Heaps exponents found here are stable across a random split of the corpus, running each test on two independent halves of the 1,984 recipes rather than the full set at once. If both halves return exponents close to the values found on the full corpus, that is a simple, self-contained way to confirm the results are not being driven by a handful of unusual recipes.


## References

[1] G. Bagler, G. K. Tewari, A. R. Yadav, A. Singh, P. Bansal, U. Dargar, M. Goel, and
M. K. Sinha, "Universal statistical laws governing culinary design," arXiv:2604.28021,
2026.

[2] O. Kinouchi, R. W. Diez-Garcia, A. J. Holanda, P. Zambianchi, and A. C. Roque, "The
non-equilibrium nature of culinary evolution," arXiv:0802.4393.

[3] A. Clauset, C. R. Shalizi, and M. E. J. Newman, "Power-law distributions in
empirical data," SIAM Review, vol. 51, no. 4, pp. 661 to 703, 2009.
doi: 10.1137/070710111

[4] M. E. J. Newman, "Power laws, Pareto distributions and Zipf's law," Contemporary
Physics, vol. 46, no. 5, pp. 323 to 351, 2005. doi: 10.1080/00107510500052444

[5] H. S. Heaps, Information Retrieval: Computational and Theoretical Aspects.
Academic Press, 1978. Original formulation of vocabulary growth as a sublinear function
of corpus size.

[6] G. Altmann, "Prolegomena to Menzerath's law," in Glottometrika 2, R. Grotjahn, Ed.
Bochum: Brockmeyer, 1980, pp. 1 to 10.

[7] Y.-Y. Ahn, S. E. Ahnert, J. P. Bagrow, and A.-L. Barabasi, "Flavor network and the
principles of food pairing," Scientific Reports, vol. 1, p. 196, 2011.
doi: 10.1038/srep00196

## Graphical Abstract

The cell below combines the Zipf fit, the Heaps fit, and the formal CSN tail test into a
single composite figure for use in the repository README.

In [16]:
# Graphical abstract: three-panel composite for the README
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 6.5))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.32)
fig.patch.set_facecolor("#0f0f13")

ax_a = fig.add_subplot(gs[0, 0])
ax_a.set_facecolor(BG)
ax_a.scatter(ranks, freqs, s=8, color="#c9b99a", alpha=0.6, zorder=3)
ax_a.plot(ranks, fit_z, color="#e07b54", lw=2,
          label=f"gamma={slope_z:.2f}\nR2={r_z**2:.2f}")
ax_a.set_xscale("log"); ax_a.set_yscale("log")
ax_a.set_xlabel("Rank", color=TC, fontsize=9)
ax_a.set_ylabel("Frequency", color=TC, fontsize=9)
ax_a.set_title("(A) Zipf's Law", color=TC, fontsize=11, loc="left")
ax_a.legend(framealpha=0.2, labelcolor=TC, fontsize=8)
ax_a.tick_params(colors=TC); ax_a.spines[:].set_color(GR)
ax_a.grid(True, alpha=0.12, color="#555")

ax_b = fig.add_subplot(gs[0, 1])
ax_b.set_facecolor(BG)
ax_b.scatter(N_points, V_points, s=8, color="#7ab5d4", alpha=0.6, zorder=3)
ax_b.plot(N_points, fit_h, color="#e07b54", lw=2,
          label=f"exp={slope_h:.2f}\nR2={r_h**2:.2f}")
ax_b.set_xscale("log"); ax_b.set_yscale("log")
ax_b.set_xlabel("Recipes N", color=TC, fontsize=9)
ax_b.set_ylabel("Vocabulary V", color=TC, fontsize=9)
ax_b.set_title("(B) Heaps' Law", color=TC, fontsize=11, loc="left")
ax_b.legend(framealpha=0.2, labelcolor=TC, fontsize=8)
ax_b.tick_params(colors=TC); ax_b.spines[:].set_color(GR)
ax_b.grid(True, alpha=0.12, color="#555")

ax_c = fig.add_subplot(gs[0, 2])
ax_c.set_facecolor(BG)
ax_c.scatter(ranks_full, sorted_data, s=8, color="#c9b99a", alpha=0.4, zorder=3)
ax_c.plot(tail_ranks, fit_line, color="#e07b54", lw=2,
          label=f"alpha={best_alpha:.2f}\nxmin={best_xmin}")
ax_c.set_xscale("log"); ax_c.set_yscale("log")
ax_c.set_xlabel("Rank", color=TC, fontsize=9)
ax_c.set_ylabel("Frequency", color=TC, fontsize=9)
ax_c.set_title("(C) CSN Tail Test", color=TC, fontsize=11, loc="left")
ax_c.legend(framealpha=0.2, labelcolor=TC, fontsize=8)
ax_c.tick_params(colors=TC); ax_c.spines[:].set_color(GR)
ax_c.grid(True, alpha=0.12, color="#555")

fig.suptitle("Scaling Laws in Philippine Cuisine Ingredient Usage",
             color=TC, fontsize=13, y=1.05)
plt.savefig("figures/graphical_abstract.png", dpi=180, bbox_inches="tight", facecolor="#0f0f13")
plt.close()
print("Graphical abstract saved to figures/graphical_abstract.png")

Graphical abstract saved to figures/graphical_abstract.png
